In [1]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession

In [2]:
builder = (SparkSession.builder
           .appName("merge-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "2g")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7efb24a1-b768-4a85-a376-129386290c00;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 199ms :: artifacts dl 10ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0 

In [3]:
%load_ext sparksql_magic
# %config SparkSql.limit=2

In [4]:
rm -rf /opt/workspace/data/delta_lake/movie_and_show_titles

In [5]:
%%sparksql
DROP TABLE IF EXISTS default.movie_and_show_titles;

In [6]:
%%sparksql 
CREATE OR REPLACE TABLE default.movie_and_show_titles ( 
    show_id STRING, 
    type STRING, 
    title STRING, 
    director STRING, 
    cast STRING, 
    country STRING, 
    date_added STRING, 
    release_year STRING, 
    rating STRING, 
    duration STRING, 
    listed_in STRING, 
    description STRING  
) USING DELTA LOCATION '/opt/workspace/data/delta_lake/movie_and_show_titles'; 

In [7]:
%%sparksql
SELECT * FROM movie_and_show_titles;

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description


In [8]:
deltaTable_titles = DeltaTable.forPath(spark, '/opt/workspace/data/delta_lake/movie_and_show_titles')

In [9]:
deltaTable_titles.toDF().show(5)

+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+



In [10]:
df_netflix = spark.read.format("delta").load('/opt/workspace/data/delta_lake/netflix_titles')

In [11]:
df_netflix_deduped = df_netflix.dropDuplicates(
    ["type", "title", "director", "date_added"]
)

In [12]:
df_netflix_deduped.show()

+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+
|show_id| type|               title|            director|                cast|             country|        date_added|release_year|rating|duration|           listed_in|         description|
+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+
|  s6274|Movie|"Behind ""The Cov...|          Keiko Yagi|                null|Japan, United States|   August 25, 2017|        2015| TV-14| 105 min|Documentaries, In...|After a documenta...|
|  s2482|Movie|   #FriendButMarried|       Rako Prijanto|Adipati Dolken, V...|           Indonesia|      May 21, 2020|        2018|  TV-G| 102 min|Dramas, Internati...|Pining for his hi...|
|  s5974|Movie|               #Roxy|     Michael K

In [13]:
(
    deltaTable_titles.alias("movie_and_show_titles")
        .merge(
            df_netflix_deduped.alias("updates"),
            """
                lower(movie_and_show_titles.type) = lower(updates.type)
            AND lower(movie_and_show_titles.title) = lower(updates.title)
            AND lower(movie_and_show_titles.director) = lower(updates.director)
            AND lower(movie_and_show_titles.date_added) = lower(updates.date_added)
            """
        )
        .whenMatchedUpdate(
            set = {
                "show_id":		"updates.show_id",
                "type": 		"updates.type",
                "title": 		"updates.title",
                "director": 	"updates.director",
                "cast": 		"updates.cast",
                "country":		"updates.country",
                "date_added":	"updates.date_added",
                "release_year":	"updates.release_year",
                "rating": 		"updates.rating",
                "duration": 	"updates.duration",
                "listed_in":	"updates.listed_in",
                "description":	"updates.description"
            }
        )
        .whenNotMatchedInsert(
            values = {
                "show_id":		"updates.show_id",
                "type": 		"updates.type",
                "title": 		"updates.title",
                "director": 	"updates.director",
                "cast": 		"updates.cast",
                "country":		"updates.country",
                "date_added":	"updates.date_added",
                "release_year":	"updates.release_year",
                "rating": 		"updates.rating",
                "duration": 	"updates.duration",
                "listed_in":	"updates.listed_in",
                "description":	"updates.description"
            }
        )
        .execute()
)

In [14]:
%%sparksql
DESCRIBE HISTORY "/opt/workspace/data/delta_lake/movie_and_show_titles"

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2025-02-26 12:49:27.070000,null,null,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'predicate': '[""(((lower(type#481) = lower(type#634)) AND (lower(title#482) = lower(title#635))) AND ((lower(director#483) = lower(director#636)) AND (lower(date_added#486) = lower(date_added#639))))""]', 'notMatchedBySourcePredicates': '[]', 'notMatchedPredicates': '[{""actionType"":""insert""}]'}",null,null,null,0,Serializable,False,"{'numOutputRows': '8806', 'numTargetBytesAdded': '2048680', 'numTargetRowsInserted': '8806', 'numTargetFilesAdded': '3', 'numTargetRowsMatchedDeleted': '0', 'numTargetFilesRemoved': '0', 'numTargetRowsMatchedUpdated': '0', 'executionTimeMs': '4013', 'numTargetRowsCopied': '0', 'rewriteTimeMs': '1957', 'numTargetRowsUpdated': '0', 'numTargetRowsDeleted': '0', 'scanTimeMs': '1338', 'numSourceRows': '8806', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'numTargetBytesRemoved': '0'}",null,Apache-Spark/3.4.1 Delta-Lake/2.4.0
0,2025-02-26 12:48:58.205000,null,null,CREATE OR REPLACE TABLE,"{'description': None, 'partitionBy': '[]', 'properties': '{}', 'isManaged': 'false'}",null,null,null,null,Serializable,True,{},null,Apache-Spark/3.4.1 Delta-Lake/2.4.0


In [15]:
%%sparksql
SELECT
     show_id
    ,type
    ,title
    ,director
    ,date_added
    ,COUNT(show_id)
FROM movie_and_show_titles
GROUP BY
     show_id
    ,type
    ,title
    ,director
    ,date_added
ORDER BY
    COUNT(show_id) DESC
;

only showing top 20 row(s)


show_id,type,title,director,date_added,count(show_id)
s7517,Movie,Mr. Church,Bruce Beresford,"December 22, 2018",1
s11,TV Show,"Vendetta: Truth, Lies and The Mafia",,"September 24, 2021",1
s4906,Movie,The Price of Success,Teddy Lussi-Modeste,"April 30, 2018",1
s23,Movie,Avvai Shanmughi,K.S. Ravikumar,"September 21, 2021",1
s7110,Movie,Jaan-E-Mann: Let's Fall in Love... Again,Shirish Kunder,"December 31, 2019",1
s1620,TV Show,Oddbods,,"December 1, 2020",1
s2513,Movie,Zaki Chan,Wael Ihsan,"May 19, 2020",1
s6841,Movie,Get Santa,Christopher Smith,"December 12, 2014",1
s2835,Movie,I am Jonas,Christophe Charrier,"March 6, 2020",1
s1625,Movie,The Da Vinci Code,Ron Howard,"December 1, 2020",1


In [16]:
%%sparksql
SELECT
    *
FROM movie_and_show_titles
LIMIT 2
;

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
s1609,Movie,3 Days to Kill,McG,"Kevin Costner, Amber Heard, Hailee Steinfeld, Connie Nielsen, Tómas Lemarquis, Richard Sammel, Marc Andréoni, Bruno Ricci, Jonas Bloquet, Eriq Ebouaney","United States, France, Serbia","December 1, 2020",2014,PG-13,117 min,Action & Adventure,A terminally ill secret agent accepts a risky mission in exchange for an experimental drug that might save him – if he can survive its side effects.
s6058,Movie,A Serious Man,"Ethan Coen, Joel Coen","Michael Stuhlbarg, Richard Kind, Fred Melamed, Sari Lennick, Adam Arkin, Amy Landecker, Alan Mandell, Fyvush Finkel, Peter Breitmayer, Aaron Wolff, Jessica McManus, Brent Braunschweig","United States, United Kingdom, France","January 16, 2018",2009,R,106 min,"Comedies, Independent Movies","With every aspect of his life unraveling, a Jewish physics professor seeks out three rabbis for spiritual guidance."


In [17]:
df_titles = (
     spark.read
    .format("csv")
    .option("header", "true")
    .load("../data/titles.csv")
)

In [18]:
df_titles_deduped = df_titles.dropDuplicates(["type", "title"])

In [19]:
df_titles_deduped.createOrReplaceTempView("titles_deduped")

In [20]:
df_titles_deduped.printSchema()

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: string (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: string (nullable = true)
 |-- imdb_votes: string (nullable = true)
 |-- tmdb_popularity: string (nullable = true)
 |-- tmdb_score: string (nullable = true)



In [21]:
%%sparksql

MERGE INTO default.movie_and_show_titles
USING titles_deduped
ON
        lower(default.movie_and_show_titles.type) = lower(titles_deduped.type)
    AND lower(default.movie_and_show_titles.title) = lower(titles_deduped.title)
    AND default.movie_and_show_titles.release_year = titles_deduped.release_year

	WHEN MATCHED THEN
		UPDATE SET
			show_id 		= titles_deduped.id,
			type 			= titles_deduped.type,
			title 			= titles_deduped.title,
			-- director 		= titles_deduped.director,
			-- cast 			= titles_deduped.cast,
			country 		= titles_deduped.production_countries,
			-- date_added 		= titles_deduped.date_added,
			release_year 	= titles_deduped.release_year,
			rating 			= titles_deduped.age_certification,
			duration 		= titles_deduped.runtime,
			listed_in 		= titles_deduped.genres,
			description 	= titles_deduped.description

	WHEN NOT MATCHED THEN
		INSERT (
			show_id,	
		    type,
            title,
            -- director,
            -- cast,
            country,
            -- date_added,
            release_year,
            rating,
            duration,
            listed_in,
            description
		)
		VALUES (
			titles_deduped.id,
			titles_deduped.type,
			titles_deduped.title,
			-- titles_deduped.director,
			-- titles_deduped.cast,
			titles_deduped.production_countries,
			-- titles_deduped.date_added,
			titles_deduped.release_year,
			titles_deduped.age_certification,
			titles_deduped.runtime,
			titles_deduped.genres,
			titles_deduped.description
		)
;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5899,2221,0,3678


In [22]:
%%sparksql
DESCRIBE HISTORY "/opt/workspace/data/delta_lake/movie_and_show_titles"

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2025-02-26 12:49:37.041000,null,null,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'predicate': '[""(((lower(type#2895) = lower(type#2865)) AND (lower(title#2896) = lower(title#2864))) AND (release_year#2901 = release_year#2867))""]', 'notMatchedBySourcePredicates': '[]', 'notMatchedPredicates': '[{""actionType"":""insert""}]'}",null,null,null,1,Serializable,False,"{'numOutputRows': '12484', 'numTargetBytesAdded': '2923027', 'numTargetRowsInserted': '3678', 'numTargetFilesAdded': '4', 'numTargetRowsMatchedDeleted': '0', 'numTargetFilesRemoved': '3', 'numTargetRowsMatchedUpdated': '2221', 'executionTimeMs': '3956', 'numTargetRowsCopied': '6585', 'rewriteTimeMs': '2064', 'numTargetRowsUpdated': '2221', 'numTargetRowsDeleted': '0', 'scanTimeMs': '1459', 'numSourceRows': '5898', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'numTargetBytesRemoved': '2048680'}",null,Apache-Spark/3.4.1 Delta-Lake/2.4.0
1,2025-02-26 12:49:27.070000,null,null,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'predicate': '[""(((lower(type#481) = lower(type#634)) AND (lower(title#482) = lower(title#635))) AND ((lower(director#483) = lower(director#636)) AND (lower(date_added#486) = lower(date_added#639))))""]', 'notMatchedBySourcePredicates': '[]', 'notMatchedPredicates': '[{""actionType"":""insert""}]'}",null,null,null,0,Serializable,False,"{'numOutputRows': '8806', 'numTargetBytesAdded': '2048680', 'numTargetRowsInserted': '8806', 'numTargetFilesAdded': '3', 'numTargetRowsMatchedDeleted': '0', 'numTargetFilesRemoved': '0', 'numTargetRowsMatchedUpdated': '0', 'executionTimeMs': '4013', 'numTargetRowsCopied': '0', 'rewriteTimeMs': '1957', 'numTargetRowsUpdated': '0', 'numTargetRowsDeleted': '0', 'scanTimeMs': '1338', 'numSourceRows': '8806', 'numTargetChangeFilesAdded': '0', 'numTargetRowsNotMatchedBySourceUpdated': '0', 'numTargetRowsNotMatchedBySourceDeleted': '0', 'numTargetBytesRemoved': '0'}",null,Apache-Spark/3.4.1 Delta-Lake/2.4.0
0,2025-02-26 12:48:58.205000,null,null,CREATE OR REPLACE TABLE,"{'description': None, 'partitionBy': '[]', 'properties': '{}', 'isManaged': 'false'}",null,null,null,null,Serializable,True,{},null,Apache-Spark/3.4.1 Delta-Lake/2.4.0


In [23]:
spark.stop()